# Install libraries

In [ ]:
!pip install diffusers transformers accelerate torch torchvision torchaudio xformers moviepy imageio gradio --quiet

# Imports libraries

In [ ]:
import torch
import gc
import random
import numpy as np
import gradio as gr
import warnings
from pathlib import Path
from typing import Optional, Tuple
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler
from diffusers.utils import export_to_video

warnings.filterwarnings("ignore")

**This code sets up an environment for AI‑based video generation using a diffusion model. It imports essential libraries (PyTorch, Gradio, Diffusers, etc.), configures a diffusion pipeline with a fast scheduler (DPMSolverMultistepScheduler), and includes utilities for memory management, reproducibility, and video export. Warnings are suppressed to keep output clean.**

# CONFIGURATION

In [ ]:
# CONFIGURATION
class VideoConfig:
    MODEL_ID = "damo-vilab/text-to-video-ms-1.7b"
    DEFAULT_HEIGHT = 256
    DEFAULT_WIDTH = 256
    DEFAULT_FRAMES = 24
    DEFAULT_STEPS = 25
    DEFAULT_GUIDANCE = 11.0
    OUTPUT_DIR = Path("/content/generated_videos")
    OUTPUT_DIR.mkdir(exist_ok=True)

Yeh code ek configuration class hai jo text-to-video model (damo-vilab/text-to-video-ms-1.7b) ke liye settings define karta hai.


MODEL_ID - Kaunsa model use hoga (Hugging Face se)

DEFAULT_HEIGHT/WIDTH - Video ka resolution (256x256)

DEFAULT_FRAMES - Har video mein kitne frames (24 frames ≈ 1 sec agar 24 fps)

DEFAULT_STEPS - Sampling steps (quality vs speed)

DEFAULT_GUIDANCE - Guidance scale (text instruction follow karne ki strength)

OUTPUT_DIR - Generated videos save kahaan honge (/content/generated_videos)

Yeh class aam taur par Diffusers library ke saath use hoti hai video generation pipelines mein.

# VIDEO PIPELINE MANAGER

In [ ]:
# VIDEO PIPELINE MANAGER
class VideoGenerationEngine:
    def __init__(self):
        self.pipe = None
        self._load_model()

    def _load_model(self):
        """Load and optimize the video generation model"""
        print("Initializing Video Generation Engine...")

        self.pipe = DiffusionPipeline.from_pretrained(
            VideoConfig.MODEL_ID,
            torch_dtype=torch.float16,
            variant="fp16",
            use_safetensors=True,
            safety_checker=None,
            low_cpu_mem_usage=True,
            ignore_mismatched_sizes=True,
        )

        # Scheduler
        self.pipe.scheduler = DPMSolverMultistepScheduler.from_config(self.pipe.scheduler.config)

        # Memory Optimizations
        self.pipe.enable_model_cpu_offload()
        self.pipe.enable_attention_slicing(1)
        self.pipe.vae.enable_slicing()
        self.pipe.vae.enable_tiling()

        if hasattr(self.pipe, "enable_xformers_memory_efficient_attention"):
            try:
                self.pipe.enable_xformers_memory_efficient_attention()
            except:
                pass

        print("Video Generation Engine Ready!")

    def generate(self,
                 prompt: str,
                 negative_prompt: Optional[str] = None,
                 num_frames: int = 24,
                 num_steps: int = 25,
                 guidance_scale: float = 11.0,
                 height: int = 256,
                 width: int = 256,
                 seed: int = -1) -> str:

        if seed == -1:
            seed = random.randint(0, 999999999)

        generator = torch.Generator(device="cpu").manual_seed(seed)

        # Default negative prompt
        if not negative_prompt:
            negative_prompt = (
                "shutterstock, watermark, text, logo, signature, branding, blurry, low quality, "
                "worst quality, deformed, ugly, bad anatomy, extra limbs, pixelated, artifacts, "
                "overexposed, underexposed"
            )

        # Enhanced prompt
        enhanced_prompt = self._enhance_prompt(prompt)

        print(f"Generating video | Seed: {seed} | Steps: {num_steps}")

        # Process prompt in chunks
        prompt_chunks = self._split_prompt(enhanced_prompt)

        all_frames = None

        for i, chunk in enumerate(prompt_chunks):
            print(f"Processing chunk {i+1}/{len(prompt_chunks)}...")

            with torch.inference_mode():
                output = self.pipe(
                    prompt=chunk,
                    negative_prompt=negative_prompt,
                    num_inference_steps=num_steps,
                    guidance_scale=guidance_scale,
                    num_frames=num_frames,
                    height=height,
                    width=width,
                    generator=generator,
                    output_type="np"
                )
                frames = output.frames[0]

                if all_frames is None:
                    all_frames = frames
                else:
                    all_frames = np.concatenate([all_frames, frames], axis=0)

        # Save video
        video_path = VideoConfig.OUTPUT_DIR / f"video_{seed}.mp4"
        export_to_video(all_frames, str(video_path), fps=8)

        print(f"Video saved: {video_path}")
        return str(video_path)

    def _enhance_prompt(self, prompt: str) -> str:
        """Professional prompt enhancement"""
        base = prompt.strip()
        if len(base.split()) < 15:
            base += ", cinematic lighting, smooth motion, highly detailed, vibrant colors, film grain"
        return base

    def _split_prompt(self, prompt: str, max_words: int = 85):
        """Split long prompts"""
        words = prompt.split()
        if len(words) <= max_words:
            return [prompt]
        return [' '.join(words[i:i + max_words]) for i in range(0, len(words), max_words)]


# INITIALIZE ENGINE
video_engine = VideoGenerationEngine()


**Yeh code ek Video Generation Engine hai jo text prompts se video banata hai.**

1. Model Load karta hai - Ek pre-trained diffusion model (text-to-video) load hota hai, jisme torch.float16 aur memory optimizations (CPU offload, attention slicing, tiling) use hote hain.

2. Video generate karta hai - Aap prompt dete ho (jaise "a cat walking on grass"), to yeh model frames generate karta hai aur unhe .mp4 file mein save kar deta hai.

3. Features:

   Agar prompt chhota hai, to automatically enhance karta hai (cinematic lighting, smooth motion...)

   Agar prompt bahut bada hai (85+ words), to chunks mein split karke process karta hai

   Negative prompt (jo nahi chahiye, jaise watermark, blurry) default set hai

   Seed fix kar sakte ho reproducibility ke liye

   Output fps 8, resolution 256x256 default

4. Output - Video save hota hai video_{seed}.mp4 path mein.

**Short summary**

**Yeh ek AI video generator hai jo text description se short video clips (24 frames) banata hai, memory optimization ke saath.**

# GRADIO UI

In [ ]:
# GRADIO INTERFACE
with gr.Blocks(title="Text to Video Generator", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # **Text to Video Generator** — Text to Video
    ### AI Text to Video Generator | Built for Quality & Performance
    """)

    with gr.Row():
        with gr.Column(scale=3):
            prompt = gr.Textbox(
                label="Video Description",
                placeholder="A cyberpunk girl with neon pink hair walking in rainy Tokyo street at night...",
                lines=3,
                value="A majestic white tiger running through dense snowy forest, cinematic lighting"
            )

            with gr.Accordion(" Advanced Settings", open=False):
                neg_prompt = gr.Textbox(
                    label="Negative Prompt",
                    value="shutterstock, watermark, text, logo, blurry, low quality, deformed",
                    lines=2
                )

                with gr.Row():
                    frames = gr.Slider(8, 48, value=24, step=4, label="Frames (Duration)")
                    steps = gr.Slider(15, 50, value=25, step=5, label="Quality Steps")

                with gr.Row():
                    guidance = gr.Slider(7.0, 15.0, value=11.0, step=0.5, label="Prompt Strength")
                    seed = gr.Number(label="Seed (-1 = Random)", value=-1)

                with gr.Row():
                    h = gr.Dropdown([256, 320, 384], value=256, label="Height")
                    w = gr.Dropdown([256, 320, 384], value=256, label="Width")

        with gr.Column(scale=1):
            generate_btn = gr.Button(" Generate Video", variant="primary", size="large")
            clear_btn = gr.Button(" Clear", variant="secondary")

    output_video = gr.Video(label="Generated Video", height=580)

    # ACTIONS
    generate_btn.click(
        fn=video_engine.generate,
        inputs=[prompt, neg_prompt, frames, steps, guidance, h, w, seed],
        outputs=output_video
    )

    clear_btn.click(
        fn=lambda: (None, ""),
        outputs=[output_video, prompt]
    )

    gr.Markdown("""
    ---
    **Note:** First generation takes longer. Subsequent generations are faster.
    """)

print("Video generator is Ready...")
demo.launch(share=True, debug=False)

**Yeh Gradio interface ka code hai jo Text-to-Video AI model ke liye ek web UI banata hai.**

User inputs: Video description (prompt), negative prompt, frames (duration), quality steps, guidance strength, dimensions, seed.

Generate button call karta hai video_engine.generate() function jo video banata hai.

Output: Generated video dikhata hai.

Clear button sab reset kar deta hai.

Pehli baar video banane mein time lagta hai, baad mein tez hota hai.

**Yeh ek interactive demo hai jo demo.launch() se chalta hai.**